# Experiment 5 Overfitting Diagnostics

Notebook 5 reported a regression on **Experiment 5 (Normal vs Boundary)** after a
set of pipeline changes labelled "val stability" (commit `5c8335c` + `9c90cf3`):

| Run | Val AUC | Test AUC | Val–Test gap |
|---|---|---|---|
| Previous | 0.727 | 0.633 | 0.094 |
| Current  | 0.667 | 0.561 | 0.106 |

Exp 2 (Normal vs Pure Tumour) is unaffected and Exp 3 (Slide Context) moved in the
expected direction. The regression is concentrated on the **subtle / low-signal**
boundary task, where the val–test gap (~0.10 AUC) matches the train–val gap during
training. That pattern is more consistent with **slide-level distribution shift +
lost regularisation** than with classical memorisation overfitting.

The pipeline changes that landed simultaneously were:
1. `BatchNormalization` → `LayerNormalization` in every conv block.
2. Intra-block `Dropout(0.2–0.4)` layers removed (only the final head `Dropout(0.5)` remains).
3. Shuffle re-seeded (`Generator.from_seed(42)`) and `deterministic=True` on interleave.
4. Outer `train_dataset.repeat()` removed (inner streams already repeat).

This notebook runs three targeted diagnostics on Exp 5 only:

- **Run A — Variance baseline.** Current architecture, three independent seeds. Quantifies how much of the regression is *seed noise* vs *systematic*.
- **Run C — BatchNorm revert.** Re-introduce the previous architecture (BN + intra-block dropout). If this restores ~0.72/0.63, the regression is the architecture change, not the data side.
- **Run F — Per-slide AUC diagnostic.** Compute AUC slide-by-slide for val and test sets. Reveals whether the val–test gap is driven by a few outlier slides (distribution shift) or by a global drop across all slides (overfitting / weak signal).

All experiments use the same 4-class dataset and the same chunk-split seed as notebook 5, so results are directly comparable. **No source files are modified** — model variants are registered locally in this notebook.

> **Cost note.** Each Exp 5 training run is ~10 min/epoch on a T4 GPU, typically converging in ~8 epochs. Run A alone is ~3.5 hours. Each cell can be run independently.

In [ ]:
# === SETUP: clone repo and install deps ===
from pathlib import Path
import os

REPO_URL = 'https://github.com/balintstewart77/camelyon16-pathology.git'
REPO_DIR = Path('/content/drive/MyDrive/new_work/Projects/Camelyon16/camelyon16-pathology')

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    !git clone {REPO_URL} "{REPO_DIR}"
else:
    !git -C "{REPO_DIR}" pull --ff-only
    print("Repository updated.")

os.chdir(REPO_DIR)
print("Current working directory:", os.getcwd())

!apt-get install -y openslide-tools > /dev/null 2>&1
!pip install -q -r requirements.txt

print("Project environment ready.")

In [ ]:
# === MOUNT DRIVE AND LOCATE DATASET (same as notebook 5) ===
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

expected_train = 'camelyon16_4class_stain_normalised'
expected_test = 'camelyon16_test_stain_normalised'

candidate_roots = [
    Path('/content/drive/MyDrive/camelyon16_data'),
    Path('/content/drive/MyDrive/new_work/Projects/pathovis_project/data'),
]

DATA_ROOT = None
TRAIN_PATH = None
TEST_PATH = None

for root in candidate_roots:
    train_path = root / expected_train
    test_path = root / expected_test
    if train_path.exists() and test_path.exists():
        DATA_ROOT = root
        TRAIN_PATH = train_path
        TEST_PATH = test_path
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find the CAMELYON16 dataset. See notebook 5 setup for the Drive shortcut instructions."
    )

print(f"DATA_ROOT  = {DATA_ROOT}")
print(f"TRAIN_PATH = {TRAIN_PATH}")
print(f"TEST_PATH  = {TEST_PATH}")

In [ ]:
# === IMPORTS ===
import gc
import json
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score

from config import DEFAULT_CONFIG
from src.models import run_binary_experiment, evaluate_on_test_set, load_model_metadata
from src.models.architectures import MODEL_REGISTRY

DEFAULT_CONFIG.training.normalise_patches = False
DEFAULT_CONFIG.training.val_max_samples_per_class = 4000

MODELS_DIR = Path('./models')
MODELS_DIR.mkdir(exist_ok=True)

print('Imports OK')

## Model variants

Three builders, all matching the `subtle` topology (3×3 kernels, 32→64→128→256
filters, stride 1 on the first conv, GAP head). They differ only in normalisation
and dropout placement, isolating the architectural changes from the data pipeline.

| Variant | Norm | Intra-block dropout | Notes |
|---|---|---|---|
| `subtle_current` | LayerNorm | none | Re-registration of the current `subtle` for clarity. Run A baseline. |
| `subtle_bn` | BatchNorm | 0.2 / 0.3 / 0.3 / 0.4 | Reverts to the pre-stability-fix architecture (Run C). |
| `subtle_lndrop` | LayerNorm | 0.2 / 0.3 / 0.3 / 0.4 | Isolates the effect of restoring dropout *only*. |

In [ ]:
def _subtle_backbone(inputs, norm_layer, dropouts):
    """Shared backbone. `norm_layer` is a callable; `dropouts` is a 4-tuple (or all None)."""
    x = layers.Conv2D(32, 3, strides=1, padding='same')(inputs)
    x = norm_layer()(x); x = layers.Activation('relu')(x)
    if dropouts[0] is not None:
        x = layers.Dropout(dropouts[0])(x)

    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)
    x = norm_layer()(x); x = layers.Activation('relu')(x)
    if dropouts[1] is not None:
        x = layers.Dropout(dropouts[1])(x)

    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = norm_layer()(x); x = layers.Activation('relu')(x)
    if dropouts[2] is not None:
        x = layers.Dropout(dropouts[2])(x)

    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = norm_layer()(x); x = layers.Activation('relu')(x)
    if dropouts[3] is not None:
        x = layers.Dropout(dropouts[3])(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    return layers.Dense(1, activation='sigmoid')(x)


def build_subtle_current(input_shape=(224, 224, 3)):
    inputs = keras.Input(shape=input_shape)
    outputs = _subtle_backbone(inputs, layers.LayerNormalization, (None, None, None, None))
    return keras.Model(inputs, outputs, name='subtle_current')


def build_subtle_bn(input_shape=(224, 224, 3)):
    inputs = keras.Input(shape=input_shape)
    outputs = _subtle_backbone(inputs, layers.BatchNormalization, (0.2, 0.3, 0.3, 0.4))
    return keras.Model(inputs, outputs, name='subtle_bn')


def build_subtle_lndrop(input_shape=(224, 224, 3)):
    inputs = keras.Input(shape=input_shape)
    outputs = _subtle_backbone(inputs, layers.LayerNormalization, (0.2, 0.3, 0.3, 0.4))
    return keras.Model(inputs, outputs, name='subtle_lndrop')


MODEL_REGISTRY['subtle_current'] = build_subtle_current
MODEL_REGISTRY['subtle_bn'] = build_subtle_bn
MODEL_REGISTRY['subtle_lndrop'] = build_subtle_lndrop

for name in ['subtle_current', 'subtle_bn', 'subtle_lndrop']:
    m = MODEL_REGISTRY[name]()
    print(f"  {name:18s}  params = {m.count_params():>9,}")
    del m
gc.collect()

## Shared helpers

- `train_exp5(model_name, seed, tag)` runs Experiment 5 (Normal vs Boundary) with the given architecture, sets the global TF/numpy/Python seed before model construction, evaluates on validation and on the held-out test set, and renames the saved checkpoint to a variant-specific filename so subsequent runs don't overwrite it.
- `predict_chunks_with_slides(model, base_path, class_to_label)` walks a 4-class chunked dataset and returns parallel arrays of slide IDs, true labels and predicted probabilities. Used by Run F to compute per-slide AUC.

In [ ]:
EXP5_MAPPING = {0: ['normal_from_normal'], 1: ['boundary_tumor']}
EXP5_BASE_PATH = './models/normal_vs_boundary.keras'


def train_exp5(model_name, seed, tag, epochs=15, learning_rate=1e-5):
    """Train Exp 5 with the given variant and seed; keep test eval; rename checkpoint."""
    print(f"\n{'#'*60}\n# {tag}: model={model_name}, seed={seed}\n{'#'*60}")
    keras.utils.set_random_seed(seed)

    exp_result = run_binary_experiment(
        dataset_path=str(TRAIN_PATH),
        experiment_type=5,
        model_name=model_name,
        epochs=epochs,
        learning_rate=learning_rate,
    )
    val_auc = exp_result['results']['auc']
    val_acc = exp_result['results']['accuracy']
    threshold = exp_result['results']['threshold']

    test_result = evaluate_on_test_set(
        exp_result['model'], str(TEST_PATH), EXP5_MAPPING, tag,
        threshold=threshold,
        normalise=False,
    )

    tagged_keras = MODELS_DIR / f"normal_vs_boundary__{tag}.keras"
    tagged_json = tagged_keras.with_suffix('.json')
    shutil.copy(EXP5_BASE_PATH, tagged_keras)
    shutil.copy(Path(EXP5_BASE_PATH).with_suffix('.json'), tagged_json)
    print(f"Saved variant checkpoint to {tagged_keras}")

    keras.backend.clear_session()
    gc.collect()

    return {
        'tag': tag,
        'model_name': model_name,
        'seed': seed,
        'val_auc': val_auc,
        'val_accuracy': val_acc,
        'threshold': threshold,
        'test_auc': test_result['auc'],
        'test_accuracy': test_result['accuracy'],
        'gap': val_auc - test_result['auc'],
    }


def predict_chunks_with_slides(model, base_path, class_to_label, batch_size=64, normalise=False):
    """Walk a 4-class chunked dataset, predict, and return (slides, y_true, y_prob) arrays.

    class_to_label: {'normal_from_normal': 0, 'boundary_tumor': 1, ...}
    Only classes listed in the dict are read.
    """
    base = Path(base_path)
    slides_out, y_true_out, y_prob_out = [], [], []

    for class_name, binary_label in class_to_label.items():
        class_dir = base / class_name
        chunk_files = sorted(class_dir.glob('*.npz'))
        print(f"  {class_name}: {len(chunk_files)} chunks")

        for chunk_path in chunk_files:
            with np.load(chunk_path) as data:
                X = data['X'].astype(np.float32)
                if X.max() > 1.5:
                    X = X / 255.0
                X = np.clip(X, 0.0, 1.0)
                if normalise:
                    mu = X.mean(axis=(1, 2), keepdims=True)
                    sd = X.std(axis=(1, 2), keepdims=True) + 1e-7
                    X = (X - mu) / sd

                slides = data['slides'] if 'slides' in data.files else np.array([b'unknown'] * len(X))
                slides = np.array([s.decode() if isinstance(s, (bytes, np.bytes_)) else str(s) for s in slides])

                probs = []
                for start in range(0, len(X), batch_size):
                    end = min(start + batch_size, len(X))
                    probs.append(model.predict(X[start:end], verbose=0).flatten())
                probs = np.concatenate(probs)

            slides_out.append(slides)
            y_true_out.append(np.full(len(probs), binary_label, dtype=np.int32))
            y_prob_out.append(probs)

    return (
        np.concatenate(slides_out),
        np.concatenate(y_true_out),
        np.concatenate(y_prob_out),
    )

## Run A — Variance baseline (current architecture × 3 seeds)

**Question.** How much of the regression is seed noise vs systematic?

**Setup.** Same architecture (`subtle_current` = current LayerNorm, no intra-block dropout). Same chunk-split seed (the train/val split via `train_test_split(random_state=42)` is hardcoded and unchanged across runs, so all seeds see identical val slides). What varies is the model-init seed via `keras.utils.set_random_seed`, which controls Conv weight initialisation, Dropout masks, and any non-pipeline TF random ops.

**Reading the result.**
- If val/test AUCs cluster tightly (std < 0.02): the regression is systematic, not noise.
- If they span the previous 0.72/0.66 numbers: a large chunk of the apparent drop is run-to-run variance and the architecture is not the dominant cause.

> Three seeds is the minimum to read a std, not a definitive answer. With ~10 min/epoch × ~8 epochs × 3 seeds, this cell takes ~3–4 hours.

In [ ]:
run_a_results = []
for seed in [42, 7, 2026]:
    run_a_results.append(
        train_exp5('subtle_current', seed=seed, tag=f'A_current_seed{seed}')
    )

print('\n=== Run A summary ===')
for r in run_a_results:
    print(f"  seed={r['seed']:>4}  val AUC={r['val_auc']:.3f}  test AUC={r['test_auc']:.3f}  gap={r['gap']:+.3f}")

val_aucs = np.array([r['val_auc'] for r in run_a_results])
test_aucs = np.array([r['test_auc'] for r in run_a_results])
print(f"  mean(val)  = {val_aucs.mean():.3f}  std = {val_aucs.std(ddof=1):.3f}")
print(f"  mean(test) = {test_aucs.mean():.3f}  std = {test_aucs.std(ddof=1):.3f}")

In [ ]:
# Visualise Run A: val vs test per seed, with previous-run reference lines
if run_a_results:
    seeds = [r['seed'] for r in run_a_results]
    vals = [r['val_auc'] for r in run_a_results]
    tests = [r['test_auc'] for r in run_a_results]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(seeds))
    w = 0.35
    ax.bar(x - w/2, vals, w, label='Val AUC', color='steelblue')
    ax.bar(x + w/2, tests, w, label='Test AUC', color='darkorange')
    ax.axhline(0.727, color='steelblue', ls='--', alpha=0.5, label='Prev val (0.727)')
    ax.axhline(0.633, color='darkorange', ls='--', alpha=0.5, label='Prev test (0.633)')
    ax.axhline(0.5, color='gray', ls=':', alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels([f'seed={s}' for s in seeds])
    ax.set_ylim(0.4, 0.85)
    ax.set_ylabel('AUC')
    ax.set_title('Run A: current architecture across seeds')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout(); plt.show()

## Run C — Restore BatchNorm + intra-block dropout

**Question.** Does reverting the architecture to its pre-stability-fix form reproduce the previous ~0.72/0.63 numbers?

**Setup.** `subtle_bn` is the current backbone with `BatchNormalization` instead of `LayerNormalization` and `Dropout(0.2/0.3/0.3/0.4)` after each conv block — i.e. the architecture *before* commit `5c8335c`. Same chunk split, same data pipeline, single seed.

**Reading the result.**
- Val ~0.72 and test ~0.63: the regression is the architecture change. The follow-up question becomes *which part* (BN vs dropout) — `subtle_lndrop` isolates that if needed.
- Val/test similar to current: the apparent regression survived the architecture revert, so the cause is elsewhere (data pipeline shuffle, slide split, seed).

In [ ]:
run_c_result = train_exp5('subtle_bn', seed=42, tag='C_bn_seed42')

print('\n=== Run C summary ===')
print(f"  val AUC = {run_c_result['val_auc']:.3f}")
print(f"  test AUC= {run_c_result['test_auc']:.3f}")
print(f"  gap     = {run_c_result['gap']:+.3f}")
print(f"  Previous: val=0.727 / test=0.633 / gap=0.094")
print(f"  Current : val=0.667 / test=0.561 / gap=0.106")

### Optional: `subtle_lndrop` to disentangle norm vs dropout

Only run this if Run C reproduces ~0.72/0.63. If LayerNorm + intra-block dropout matches Run C, the missing regulariser was the dropout layers (and we can keep LayerNorm). If it lands closer to the current numbers, the choice of normalisation matters too.

In [ ]:
# Uncomment to run
# run_b_result = train_exp5('subtle_lndrop', seed=42, tag='B_lndrop_seed42')
# print(f"  val AUC = {run_b_result['val_auc']:.3f}  test AUC = {run_b_result['test_auc']:.3f}")

## Run F — Per-slide AUC diagnostic

**Question.** Is the val–test gap driven by a few outlier slides (distribution shift), or by a global drop across most slides (overfitting / weak signal)?

**Method.** For a chosen trained model, walk the val *and* test chunks of the 4-class dataset directly (preserving slide IDs from the chunk metadata), produce predictions, and compute AUC per slide for any slide that has at least 20 patches of each class. The val set draws from train slides (held-out chunks); the test set is entirely separate slides.

**Reading the result.**
- **Heavy-tailed test distribution with a few low-AUC slides.** Distribution shift / appearance heterogeneity is dominating; the model works on most slides but a handful drag the aggregate down. Look for stain or scanner outliers.
- **Uniform drop across slides.** No single villain; the signal is just weak. Regularisation and augmentation are the lever.
- **Val and test distributions look similar.** The apparent val–test gap is largely chunk-/aggregation-level noise rather than a per-slide story.

> The model evaluated here is whichever variant you point `EVAL_MODEL_PATH` at. Default uses Run A seed=42; change to a Run C path to inspect the BN variant.

In [ ]:
# Pick which trained variant to diagnose
EVAL_MODEL_PATH = './models/normal_vs_boundary__A_current_seed42.keras'
# EVAL_MODEL_PATH = './models/normal_vs_boundary__C_bn_seed42.keras'

# Reproduce the val/train chunk split used during training so we evaluate on the
# *same* val slides the model saw, but reading slide IDs directly from each chunk.
from sklearn.model_selection import train_test_split
from src.dataset.tf_pipeline import load_chunk_paths, create_binary_dataset, verify_no_slide_leakage

binary_path = create_binary_dataset(str(TRAIN_PATH), EXP5_MAPPING, 'F_diagnostic')
try:
    all_chunks = load_chunk_paths(binary_path)
    train_chunks, val_chunks = train_test_split(
        all_chunks, test_size=DEFAULT_CONFIG.training.val_split,
        random_state=42, stratify=[lbl for _, lbl in all_chunks]
    )
    val_files = [f for f, _ in val_chunks]
    verify_no_slide_leakage([f for f, _ in train_chunks], val_files)
    print(f'Val chunks: {len(val_files)}')

    eval_model = keras.models.load_model(EVAL_MODEL_PATH)
    print(f'Loaded model: {EVAL_MODEL_PATH}')

    # Walk val chunks (binary structure: normal / tumor subfolders) with slide IDs
    val_slides, val_y, val_p = [], [], []
    for chunk_path in val_files:
        cp = Path(chunk_path)
        label = 0 if cp.parent.name == 'normal' else 1
        with np.load(cp) as data:
            X = data['X'].astype(np.float32)
            if X.max() > 1.5: X = X / 255.0
            X = np.clip(X, 0.0, 1.0)
            slides = data['slides'] if 'slides' in data.files else np.array([b'unknown'] * len(X))
            slides = np.array([s.decode() if isinstance(s, (bytes, np.bytes_)) else str(s) for s in slides])

            probs = []
            for start in range(0, len(X), 64):
                end = min(start + 64, len(X))
                probs.append(eval_model.predict(X[start:end], verbose=0).flatten())
            probs = np.concatenate(probs)

        val_slides.append(slides)
        val_y.append(np.full(len(probs), label, dtype=np.int32))
        val_p.append(probs)
    val_slides = np.concatenate(val_slides); val_y = np.concatenate(val_y); val_p = np.concatenate(val_p)
    print(f'Val patches: {len(val_y):,} across {len(np.unique(val_slides))} slides')
finally:
    try: shutil.rmtree(binary_path)
    except Exception: pass

# Test set: walk the 4-class test dataset directly (no train/val split)
print('\nWalking test chunks...')
test_slides, test_y, test_p = predict_chunks_with_slides(
    eval_model, str(TEST_PATH), {'normal_from_normal': 0, 'boundary_tumor': 1}
)
print(f'Test patches: {len(test_y):,} across {len(np.unique(test_slides))} slides')

In [ ]:
def per_slide_auc(slides, y_true, y_prob, min_per_class=20):
    rows = []
    for s in np.unique(slides):
        mask = slides == s
        yt, yp = y_true[mask], y_prob[mask]
        n_pos = int((yt == 1).sum()); n_neg = int((yt == 0).sum())
        if n_pos >= min_per_class and n_neg >= min_per_class:
            rows.append((s, roc_auc_score(yt, yp), n_pos + n_neg, n_pos, n_neg))
    return rows


val_per_slide = per_slide_auc(val_slides, val_y, val_p)
test_per_slide = per_slide_auc(test_slides, test_y, test_p)

print(f'Val slides w/ both classes (≥20 each): {len(val_per_slide)}')
print(f'Test slides w/ both classes (≥20 each): {len(test_per_slide)}')

val_aucs_ps = np.array([r[1] for r in val_per_slide]) if val_per_slide else np.array([])
test_aucs_ps = np.array([r[1] for r in test_per_slide]) if test_per_slide else np.array([])

if len(val_aucs_ps): print(f'  val  per-slide AUC: median={np.median(val_aucs_ps):.3f}  mean={val_aucs_ps.mean():.3f}  std={val_aucs_ps.std():.3f}  min={val_aucs_ps.min():.3f}')
if len(test_aucs_ps): print(f'  test per-slide AUC: median={np.median(test_aucs_ps):.3f}  mean={test_aucs_ps.mean():.3f}  std={test_aucs_ps.std():.3f}  min={test_aucs_ps.min():.3f}')

# Worst 5 test slides
if test_per_slide:
    print('\nWorst 5 test slides:')
    for s, auc, n, npos, nneg in sorted(test_per_slide, key=lambda r: r[1])[:5]:
        print(f'  {s:20s}  AUC={auc:.3f}  n={n} (pos={npos}, neg={nneg})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

if len(val_aucs_ps) and len(test_aucs_ps):
    bins = np.linspace(0.0, 1.0, 21)
    axes[0].hist(val_aucs_ps, bins=bins, alpha=0.7, label=f'Val (n={len(val_aucs_ps)})', color='steelblue')
    axes[0].hist(test_aucs_ps, bins=bins, alpha=0.7, label=f'Test (n={len(test_aucs_ps)})', color='darkorange')
    axes[0].axvline(0.5, color='gray', ls=':')
    axes[0].set_xlabel('Per-slide AUC'); axes[0].set_ylabel('Slides')
    axes[0].set_title('Per-slide AUC distribution')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    parts = axes[1].boxplot([val_aucs_ps, test_aucs_ps], labels=['Val', 'Test'], patch_artist=True)
    for p, c in zip(parts['boxes'], ['steelblue', 'darkorange']):
        p.set_facecolor(c); p.set_alpha(0.7)
    axes[1].axhline(0.5, color='gray', ls=':')
    axes[1].set_ylabel('Per-slide AUC'); axes[1].set_title('Per-slide AUC by split')
    axes[1].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'Not enough slides with both classes', ha='center', va='center')

plt.tight_layout(); plt.show()

## Interpretation — fill in after running

Once Runs A, C and F have completed, the conclusion should be one of:

1. **Mostly seed noise.** Run A std is large (~0.05+) and at least one seed lands near the previous numbers. Action: re-baseline by reporting mean ± std across seeds rather than a single run; revisit only if the mean is meaningfully worse.
2. **Architecture regression.** Run A clusters around the new numbers; Run C reproduces the previous numbers. Action: decide which regulariser to restore (BN, intra-block dropout, or both via `subtle_lndrop`).
3. **Distribution shift.** Run F shows a heavy-tailed test distribution with a few low-AUC slides driving the aggregate. Action: investigate those slides (stain, scanner, tumour type); consider stain augmentation rather than touching the architecture.
4. **Weak signal.** Per-slide AUCs are uniformly mediocre on both splits. Action: stronger regularisation, augmentation, or accept that Exp 5 has an intrinsic ceiling at this model size.